# c4fairness — worked example (binary classification, COMPAS)

`c4fairness` clusters the rows of a model's test set and measures how **error
disparities** vary across the discovered clusters and demographic groups. Here we
audit a recidivism classifier on **COMPAS** (`Data/compas/Compas_error_shap.csv`)
via the Python API — no R required.

Sensitive features cover all three kinds: **binary** (`sex`), **multi-categorical**
(`race`), **numeric** (`age`).

In [ ]:
import pandas as pd
from c4fairness.preprocessing import encode_categoricals
from c4fairness.clustering import cluster
from c4fairness.fairness_metrics import binary_error_rate_column
from c4fairness.cli import _build_sensitive_analysis_list, apply_salient_reconstruction
from c4fairness.experiments import make_recap
from c4fairness.result_viz import plot_cluster_recap_heatmap
from IPython.display import Image

raw = pd.read_csv("../Data/compas/Compas_error_shap.csv")
# The file ships pre-made scaled/one-hot/SHAP columns; keep only the readable ones so
# encode_categoricals can one-hot 'race'/'sex' itself without name collisions.
df = raw[["age", "priors_count", "sex", "race", "true_class", "predicted_class"]].copy()
df.head()

## 1. Columns

`regular` features drive the clustering geometry; `sensitive` are the protected
attributes we audit. `age` is declared **continuous** (analysed by median).

In [ ]:
regular   = ["age", "priors_count"]
sensitive = ["sex", "race", "age"]          # binary, multi-categorical, numeric
col_lists = {"regular": regular, "sensitive": sensitive, "proxy": [], "special": []}
orig_sensitive = list(sensitive)

## 2. Encode + cluster

`encode_categoricals` one-hot-encodes `race` (and binary `sex`) for the Euclidean
distance and returns `multiclass_dummies` (mcd) so the tables can rebuild the readable
category. We fix `k=4`.

In [ ]:
dfe, cl, cat_names, mcd, ohe = encode_categoricals(
    df.copy(), col_lists, [], "kmeans", distance="euclidean"
)
clustering_cols = cl["regular"] + cl["sensitive"]
res = cluster(dfe[clustering_cols], algorithm="kmeans", distance="euclidean",
              n_clusters=4, random_state=42)
print("clusters:", res.n_clusters, "| silhouette:", round(res.silhouette, 3))
print("sizes:", res.cluster_sizes)

## 3. Error metric + recap

We audit the **false-positive rate** (`fpr = FP/(FP+TN)`) — labelling a defendant
high-risk who did not re-offend — derived per row from `true_class` / `predicted_class`.
Sensitive features are shown in **salient** form (readable winning category per feature).

In [ ]:
analysis = _build_sensitive_analysis_list(cl["sensitive"], mcd, orig_sensitive, option="salient")

dfe["fpr"] = binary_error_rate_column(df["true_class"], df["predicted_class"], "fpr").values
res_df = dfe.copy()
res_df["clusters"] = res.labels
apply_salient_reconstruction(res_df, mcd, orig_sensitive)   # rebuild readable 'race'

recap = make_recap(res_df, clustering_cols, sensitive_cols=analysis,
                   error_col="fpr", error_type="binary",
                   feature_matrix=res.feature_matrix,
                   continuous_sensitive_cols=["age"])
recap.round(3)

`error_value` is each cluster's FPR; `error_gap` its one-vs-all gap with Fisher
significance `error_gap_sig`; `race_cat` names the dominant race, `sex_Male_value` the
male proportion, `age_value` the median age.

## 4. Heatmap

In [ ]:
plot_cluster_recap_heatmap(recap.copy(), "compas_fpr", ".", error_label="FP Rate")
Image("compas_fpr.png")

## Takeaway

The cluster dominated by **African-American** defendants carries the highest FPR with a
significant `FP Rate gap sig.` — the model flags them as high-risk-but-non-reoffending
more often than other groups. This reproduces the well-known COMPAS false-positive
disparity, now localized to a specific subgroup by unsupervised clustering. Swap `"fpr"`
for `"fnr"` to audit the opposite error direction.